# IL3.2: Análisis de Trazabilidad y Logs
## Notebook 5: Trazabilidad avanzada: métricas, anomalías y conexión con herramientas LLMOps

### Objetivo:
Aprender a explotar la capa de datos de logs para transformarla en información de valor para el negocio y la ingeniería de software. El estudiante utilizará visualizaciones gráficas con `matplotlib` para identificar anomalías (como peticiones demasiado lentas) y establecerá la conexión conceptual entre estos scripts caseros y las plataformas profesionales de LLMOps del mercado.

### ¿Qué es una Anomalía en LLMs?
1. **Latencias Extremas:** Respuestas que tardan mucho más del promedio, lo que arruina la experiencia del usuario.
2. **Errores Frecuentes:** Picos de fallos generados por fallos en las APIs o entradas imprevistas.
3. **Uso Excesivo de Recursos:** Consultas que consumen miles de tokens innecesariamente (ej. loops infinitos del agente).
4. **Bajo Nivel de Confianza Consistente:** Indicativo de que el prompt o el modelo no están alineaos con el dominio.


### Inicialización del Entorno
Ejecuta la siguiente celda para configurar el cliente LLM y el agente real de Wikipedia usando LangChain.


In [ ]:
import os
import wikipedia
from langchain_openai import ChatOpenAI

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM
try:
    llm = ChatOpenAI(
        model="gpt-4o",
        openai_api_base=os.environ.get("GITHUB_BASE_URL"),
        openai_api_key=os.environ.get("GITHUB_TOKEN"),
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

from langchain_classic.agents import tool, create_openai_tools_agent, AgentExecutor
from langsmith import Client

@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Útil para obtener información sobre personas, lugares o conceptos."""
    try:
        return wikipedia.summary(query, sentences=2)
    except Exception as e:
        return f"Ocurrió un error: {e}"

tools = [get_wikipedia_summary]

client = Client(None)
# Usamos el mismo prompt de la comunidad que ya está preparado para manejar historial
prompt = client.pull_prompt("hwchase17/openai-tools-agent", dangerously_pull_public_prompt=True)

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✅ Agente y herramientas listos.")


### Generación de Logs Reales/Simulados
Para realizar un análisis de tendencias y latencias realista, ejecutaremos un set de 20 consultas automatizadas al agente de Wikipedia (usando la función estructurada del Notebook 3). Algunas de estas consultas invocarán la herramienta y otras se responderán directamente por el LLM.


In [ ]:
import pandas as pd
import json
import numpy as np
import matplotlib.pyplot as plt
import time
import random
from datetime import datetime
from langchain_core.callbacks import BaseCallbackHandler

log_file = "agent_historical_real.jsonl"

# Inicializar limpio
with open(log_file, "w", encoding="utf-8") as f:
    pass

# Lista de consultas variadas
test_queries = [
    "¿Quién fue Albert Einstein?",
    "Hola agente, ¿cómo te llamas?",
    "¿Cuál es la capital de Francia?",
    "Cuéntame una broma corta.",
    "Busca información de Alan Turing en Wikipedia.",
    "Dame la definición de fotosíntesis.",
    "¿Qué hora es?",
    "¿Quién pintó la Mona Lisa?",
    "Dame un saludo cordial.",
    "¿Cuál es la distancia de la tierra al sol?",
    "Dime quién fue Steve Jobs.",
    "¿Qué es un algoritmo?",
    "Buscar datos históricos sobre el Imperio Romano.",
    "¿Cómo estás?",
    "¿Quién escribió Hamlet?",
]

def run_structured_collection(user_id, query):
    trace_id = str(uuid.uuid4())
    start_time = time.time()
    
    tool_used = None
    success = True
    error_type = None
    
    class Tracker(BaseCallbackHandler):
        def on_tool_start(self, serialized, input_str, **kwargs):
            nonlocal tool_used
            tool_used = serialized.get("name", "get_wikipedia_summary")
            
    try:
        if llm is None:
            # Simulación realista de tiempos de respuesta
            time.sleep(random.uniform(1.2, 2.5) if ("quién" in query.lower() or "busca" in query.lower()) else random.uniform(0.15, 0.4))
            tool_used = "get_wikipedia_summary" if ("quién" in query.lower() or "busca" in query.lower()) else None
            response_text = f"Simulado: {query}"
        else:
            response = agent_executor.invoke({"input": query}, config={"callbacks": [Tracker]})
            response_text = response.get("output", "")
    except Exception as e:
        success = False
        error_type = type(e).__name__
        response_text = "Fallo."
        
    duration = round(time.time() - start_time, 4)
    tokens_est = (len(query) + len(response_text)) // 4
    
    entry = {
        "timestamp": datetime.now().isoformat(),
        "trace_id": trace_id,
        "user_id": user_id,
        "query": query,
        "response": response_text,
        "response_time": duration,
        "tokens_used": tokens_est,
        "success": success,
        "tool_used": tool_used,
        "error_type": error_type
    }
    
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

import uuid
print("Recolectando métricas reales de ejecución de consultas...")
for i, q in enumerate(test_queries):
    user_id = f"user_{random.randint(1, 5)}"
    run_structured_collection(user_id, q)
print("Recolección completada. Datos guardados en 'agent_historical_real.jsonl'.")


### Análisis de Anomalías de Latencia
Cargaremos los logs reales en un DataFrame de `pandas` y calcularemos el umbral estadístico de anomalías de lentitud utilizando la desviación estándar.


In [ ]:
# Carga en pandas
rows = []
with open("agent_historical_real.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))
            
df = pd.DataFrame(rows)

# Cálculo estadístico
avg_latency = df["response_time"].mean()
std_latency = df["response_time"].std()
# Umbral: Latencias mayores a la media + 1.5 veces la desviación estándar
anomaly_threshold = avg_latency + 1.5 * std_latency

print(f"Latencia promedio: {avg_latency:.4f} segundos")
print(f"Desviación estándar: {std_latency:.4f} segundos")
print(f"Umbral de anomalía (Media + 1.5*Std): {anomaly_threshold:.4f} segundos")

anomalies = df[df["response_time"] > anomaly_threshold]
print(f"\n--- Peticiones Lentas Detectadas ({len(anomalies)}) ---")
print(anomalies[["query", "response_time", "tool_used"]])


### Visualización Gráfica
Generaremos histogramas y gráficos de dispersión para correlacionar visualmente si el uso de la herramienta de Wikipedia influye directamente en las latencias del agente real.


In [ ]:
# Gráfico 1: Histograma de Latencias
plt.figure(figsize=(10, 4))
plt.hist(df["response_time"], bins=8, color="lightcoral", edgecolor="black", alpha=0.7)
plt.axvline(avg_latency, color="blue", linestyle="dashed", linewidth=1.5, label=f"Media: {avg_latency:.2f}s")
plt.axvline(anomaly_threshold, color="red", linestyle="dotted", linewidth=1.5, label=f"Umbral Anomalía: {anomaly_threshold:.2f}s")
plt.title("Distribución de Latencias en el Agente Real")
plt.xlabel("Tiempo de Respuesta (s)")
plt.ylabel("Frecuencia")
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

# Gráfico 2: Latencia comparando Con vs Sin Herramienta
df_with_tool = df[df["tool_used"].notnull()]
df_no_tool = df[df["tool_used"].isnull()]

plt.figure(figsize=(8, 4))
plt.bar(["Sin Herramienta (Directo)", "Con Wikipedia"], 
        [df_no_tool["response_time"].mean(), df_with_tool["response_time"].mean()], 
        color=["skyblue", "lightgreen"], edgecolor="black")
plt.ylabel("Tiempo Promedio (s)")
plt.title("Impacto del uso de Wikipedia en la Latencia del Agente")
plt.grid(axis='y', alpha=0.3)
plt.show()


### Conexión conceptual con Herramientas de la Industria (LLMOps)
En un sistema de producción real, no analizamos los logs manualmente con scripts de Python para cada servidor. En su lugar, utilizamos plataformas de **LLMOps** y herramientas de observabilidad:
- **LangSmith / Langfuse:** Diseñadas específicamente para trazar el flujo completo de cadenas (chains), llamadas a LLM y ejecución de herramientas de agentes (LangChain). Permiten visualizar grafos de llamadas, calcular costos por paso y depurar prompts.
- **OpenTelemetry / Prometheus / Grafana:** Utilizados para el monitoreo clásico de infraestructura de software. Permiten definir métricas personalizadas como "latencia del modelo", "errores HTTP" y generar alertas de producción.
- **MLflow / WandB:** Enfoques centrados en el ciclo de vida del modelo y experimentos de optimización de prompts y fine-tuning.


### 🛠️ Reto Práctico (Mini-entrega)

**Instrucciones:**
1. Escribe un script en Python que analice el archivo de logs estructurados históricos (`agent_historical_real.jsonl`).
2. El script debe:
   - Calcular la tasa de éxito general del agente.
   - Guardar un archivo PNG llamado `reporte_latencia_wikipedia.png` que contenga un diagrama de caja (boxplot) de las latencias del agente agrupado por el campo `tool_used`.
   - Mostrar el gráfico en pantalla y verificar que la imagen se guardó correctamente en el disco.


In [ ]:
# Desarrolla tu solución aquí

# 1. Calcular tasa de éxito

# 2. Generar y guardar gráfico boxplot

# 3. Mostrar y verificar guardado del archivo reporte_latencia_wikipedia.png


### 📝 Preguntas de Análisis
1. **En tu reto práctico, ¿cómo se diferencia la latencia cuando el agente requiere llamar a la herramienta externa (Wikipedia) comparada con cuando responde directamente? ¿Por qué ocurre esto?**
2. **Explica la diferencia conceptual entre monitorear un microservicio REST tradicional (ej. latencia HTTP, uso de CPU/RAM) y hacer observabilidad a un agente inteligente con LLMs.**
3. **Investiga qué es LangSmith o Langfuse. ¿Qué funcionalidad clave de estas plataformas resuelve el problema de tener que programar trazas a mano como hicimos en el Notebook 2?**
